[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxischa/datacamp_test/blob/main/bloc4_ml/cours/seance4_cours.ipynb)

# Séance 4.4 — Segmenter sans étiquette — quatre clients, quatre traitements

**Cours** · durée : 2h (≈70 min de cours, ≈50 min d'exercices)

> ⚠️ **Avant de taper quoi que ce soit :** *Fichier → Enregistrer une copie dans Drive*. Sinon votre travail sera perdu en fermant l'onglet.
>
> 📱 Sur tablette, faites d'abord les réglages de [Bien démarrer](https://github.com/maxischa/datacamp_test/blob/main/ressources/setup_tablette.md).

## Objectifs

À la fin de cette séance, vous saurez :

- distinguer un problème supervisé d'un problème non supervisé
- dire ce que `KMeans` cherche à minimiser, et lire une inertie et une silhouette
- expliquer pourquoi il faut standardiser avant de mesurer une distance
- repérer deux variables qui disent la même chose, et ce que ça coûte
- donner un nom et un chiffre d'affaires à chaque segment obtenu
- transformer une segmentation en plan d'action budgété

## Plus de cible

Depuis la séance 4.1, chaque problème avait un `y` : un montant à prévoir, un
départ à anticiper. On pouvait donc mesurer si on avait raison.

Aujourd'hui, on retourne chez le détaillant des blocs 2 et 3, et la question
change de nature :

> *« Combien de types de clients avons-nous, et à quoi ressemblent-ils ? »*

Personne n'a étiqueté ces clients. C'est de l'apprentissage **non supervisé** —
et il n'y a **pas de bonne réponse**. Il y a des découpages plus ou moins
utiles, ce qui n'est pas la même chose.

On va y venir. Mais pas tout de suite : d'abord cinq minutes sur un cas
fabriqué, où l'on connaît la réponse.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.datasets import make_blobs

# Affichage adapte aux petits ecrans
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 80)

# Les donnees sont lues directement depuis le web : rien a telecharger
BASE = "https://raw.githubusercontent.com/maxischa/datacamp_test/main/bloc4_ml/data/"

## 1. Un cas d'école — que cherche `KMeans`, au juste ?

Avant les vrais clients, cinq minutes sur des données **fabriquées exprès** :
300 individus, deux variables, trois groupes qu'on voit à l'œil nu.

L'intérêt est qu'ici **on connaît la réponse à l'avance**. On peut donc
vérifier que les indicateurs la retrouvent — et savoir enfin à quoi ressemble
un bon score.

In [ ]:
# make_blobs fabrique des groupes sur commande : ici 3, bien separes
X_ecole, _ = make_blobs(n_samples=300, centers=3, cluster_std=0.8,
                        n_features=2, random_state=42)

plt.figure(figsize=(7, 4))
plt.scatter(X_ecole[:, 0], X_ecole[:, 1], alpha=0.6)
plt.title("Trois groupes, visibles a l'oeil nu")
plt.xlabel("variable 1")
plt.ylabel("variable 2")
plt.show()

### Ce que l'algorithme optimise

`KMeans` cherche **`k` centres** et affecte chaque point au centre le plus
proche, de façon à rendre aussi petite que possible une seule quantité :

> **la somme des carrés des distances de chaque point au centre de son groupe.**

Cette quantité porte un nom : c'est l'**inertie**. Retenez la conséquence,
elle explique tout le reste de la séance :

> ⚠️ **L'inertie n'est pas un diagnostic ajouté après coup — c'est exactement
> ce que l'algorithme minimise.** Lui demander si son découpage est bon revient
> à lui demander de noter sa propre copie.

L'algorithme s'y prend en trois temps, répétés jusqu'à ce que plus rien ne
bouge : placer `k` centres au hasard, affecter chaque point au plus proche,
déplacer chaque centre au milieu de ses points. Le `n_init=10` que vous verrez
partout signifie « recommence 10 fois depuis des positions de départ
différentes, et garde le meilleur essai » — sans quoi le résultat dépend du
hasard initial.

### L'inertie : les groupes sont-ils **serrés** ?

L'inertie vaut zéro si chaque point est confondu avec son centre, et elle
grandit dès qu'ils s'en éloignent.

> **Basse = groupes compacts.**

Avec un piège, et il est de taille : l'inertie **baisse toujours** quand `k`
augmente, mécaniquement. Sur nos 300 points, à `k` = 300 chaque point serait
son propre groupe, chaque distance serait nulle, et l'inertie vaudrait
**zéro**. Une inertie plus basse ne veut donc **pas** dire un meilleur
découpage — ce serait absurde.

Deux façons de la rendre lisible :

1. **La rapporter à son point de départ.** Sans aucun groupe (`k` = 1), le
   centre unique est la moyenne générale : c'est toute la dispersion du nuage.
   Chaque découpage en absorbe une part, qu'on peut exprimer en pourcentage.
2. **Chercher le coude** — jamais le minimum : l'endroit où ajouter un groupe
   cesse de faire gagner grand-chose.

In [ ]:
km3 = KMeans(n_clusters=3, n_init=10, random_state=42).fit(X_ecole)

plt.figure(figsize=(7, 4))
plt.scatter(X_ecole[:, 0], X_ecole[:, 1], c=km3.labels_, alpha=0.6)
plt.scatter(km3.cluster_centers_[:, 0], km3.cluster_centers_[:, 1],
            c="red", marker="X", s=200)   ## les trois centres trouves
plt.title(f"k = 3 : inertie = {km3.inertia_:.0f}")
plt.show()

Les trois centres tombent au milieu des trois paquets, et l'inertie vaut
**363**. Mais 363, c'est beaucoup ou c'est peu ? Seul, ce nombre ne dit rien —
et on vient de voir qu'on ne peut pas se contenter de la chercher basse.

### La silhouette : les groupes sont-ils **séparés** ?

Il nous faut un second indicateur, qui ne soit **pas** celui que l'algorithme
optimise. Pour un point donné, on calcule deux distances moyennes :

- **a** = sa distance moyenne aux **autres points de son groupe** ;
- **b** = sa distance moyenne aux points du **groupe voisin le plus proche**.

Sa silhouette vaut **(b − a) / max(a, b)**, donc toujours entre −1 et 1 :

| Valeur | Ce que ça veut dire |
|---|---|
| proche de **1** | `b` très supérieure à `a` : le point est bien plus proche des siens que du groupe d'à côté — il est **au bon endroit** |
| proche de **0** | `a ≈ b` : il est **à la frontière**, les deux groupes se valent pour lui |
| **négative** | `a > b` : il serait **mieux placé dans le groupe voisin** |

`silhouette_score` en donne la **moyenne sur tous les points**. Contrairement à
l'inertie, elle **ne bouge pas mécaniquement avec `k`** : on peut donc comparer
directement deux valeurs de `k`, et chercher un maximum.

> 💡 **C'est la différence qui compte entre les deux.** L'inertie est la copie
> de l'algorithme ; la silhouette est un correcteur extérieur.

### Les deux indicateurs, sur un cas où l'on connaît la réponse

In [ ]:
# k = 1 : un seul centre, donc toute la dispersion du nuage
totale_ecole = KMeans(n_clusters=1, n_init=10, random_state=42).fit(X_ecole).inertia_

lignes = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_ecole)
    lignes.append({"k": k, "inertie": round(km.inertia_, 1),
                   "reste_pct": round(100 * km.inertia_ / totale_ecole, 1),
                   "silhouette": round(silhouette_score(X_ecole, km.labels_), 3)})

ecole = pd.DataFrame(lignes).set_index("k")
ecole

In [ ]:
ecole["inertie"].plot(marker="o", figsize=(7, 4))
plt.title("L'inertie : on cherche le COUDE")
plt.xlabel("nombre de groupes k")
plt.ylabel("inertie restante")
plt.show()

ecole["silhouette"].plot(marker="o", figsize=(7, 4))
plt.title("La silhouette : on cherche le SOMMET")
plt.xlabel("nombre de groupes k")
plt.ylabel("silhouette moyenne")
plt.show()

### Lecture — et c'est le cas idéal

**L'inertie** s'effondre de 20 121 à 5 527 puis à **363** : à `k` = 3, il ne
reste que **1,8 %** de la dispersion de départ. Ensuite, plus rien ne bouge —
318, puis 273. Le **coude** est franc, et il est à 3. Elle continue pourtant de
baisser après 3, comme annoncé : c'est bien le coude qu'on lit, pas le minimum.

**La silhouette** culmine à **0,878** pour `k` = 3, puis chute : 0,697 à 4,
0,503 à 5. Elle, elle a un vrai maximum, parce qu'elle mesure la
**séparation** et non la compacité — découper un vrai groupe en deux rapproche
les frontières et la fait chuter.

Ce qui donne le barème à garder en tête pour toute la suite :

| Score de silhouette | Interprétation |
|---|---|
| au-delà de **0,7** | groupes nettement séparés — le cas d'école ci-dessus |
| entre **0,5 et 0,7** | structure claire |
| entre **0,3 et 0,5** | groupes réels mais qui **se chevauchent** |
| en dessous de **0,25** | pas de structure convaincante |

> 💡 **Les deux indicateurs sont d'accord ici, et pointent la bonne réponse.**
> Gardez cette image : c'est le seul moment de la séance où ce sera aussi
> simple. Sur de vrais clients, ils se contrediront — et il faudra trancher.

### Les vraies données : recency, frequency, monetary

Trois variables, un classique du marketing depuis quarante ans.

In [ ]:
cli = pd.read_csv(BASE + "clients_rfm.csv")   ## une ligne = un client

print(cli.shape)
cli.head(3)

In [ ]:
cli[["recence", "freq", "montant"]].describe().round(1)   ## les echelles

`recence` = jours depuis le dernier achat · `freq` = nombre de commandes ·
`montant` = total dépensé.

Notez les échelles : la récence va de 0 à 372, le montant de 30 à 143 825. **Un
facteur mille entre les deux.** Retenez-le, c'est le sujet du paragraphe
suivant.

## 2. Pourquoi standardiser — la démonstration

`KMeans` regroupe ce qui est **proche**. Avant de le lancer, regardons
simplement le nuage de deux de nos variables.

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(cli["freq"], cli["montant"], alpha=0.5)
plt.title("Echelles brutes : presque tout le monde est ecrase en bas a gauche")
plt.xlabel("freq : nombre de commandes")
plt.ylabel("montant total depense (euros)")
plt.show()

**Cette figure est presque vide.** Un client à 143 825 € et 201 commandes tire
seul les deux axes, et les 471 autres s'entassent dans le coin. Ce n'est pas un
défaut d'affichage : c'est exactement ce que voit `KMeans`, qui raisonne sur
ces distances-là.

Voyons ce que ça donne.

In [ ]:
# Sur les colonnes BRUTES, sans mise a l'echelle : volontairement faux
brut = KMeans(n_clusters=4, n_init=10, random_state=42).fit(cli[["recence", "freq", "montant"]])

cli.assign(g=brut.labels_).groupby("g").agg(
    n=("client_id", "size"), recence=("recence", "median"),
    freq=("freq", "median"), montant=("montant", "median")).round(1)

Regardez la colonne `n` : **401, 58, 11 et 2**. Il y a bien quatre groupes,
comme demandé — mais **un seul en contient 85 %**, et les trois autres se
partagent les miettes, dont un groupe de **deux clients**.

C'est le symptôme d'un découpage qui a échoué. Segmenter, c'est séparer une
clientèle en parts qu'on peut traiter différemment ; ici on obtient « presque
tout le monde », plus trois poignées d'exceptions. Aucune action commerciale ne
se construit là-dessus.

Pourquoi ce résultat ? Un écart de 100 000 € sur le montant écrase complètement
un écart de 300 jours sur la récence. `KMeans` additionne des euros et des
jours comme si c'était la même unité : les euros étant mille fois plus gros, ils
décident de tout. **L'algorithme n'a regardé qu'une seule variable**, et il
a séparé les quelques très gros clients du reste.

> ⚠️ Notez qu'aucun message d'erreur n'est apparu. Le code est correct, le
> résultat est faux : c'est une **erreur silencieuse**, comme celle des dates au
> bloc 2.

### Correction 1 : écraser les valeurs extrêmes

Le logarithme rapproche les grands nombres sans toucher à leur ordre. Le même
nuage, avec un axe vertical logarithmique :

In [ ]:
plt.figure(figsize=(7, 4))
plt.scatter(cli["freq"], cli["montant"], alpha=0.5)
plt.xscale("log")   ## les DEUX axes changent d'echelle
plt.yscale("log")
plt.title("Les memes clients, en echelle log sur les deux axes")
plt.xlabel("freq : nombre de commandes (echelle log)")
plt.ylabel("montant total depense (echelle log)")
plt.show()

Le nuage occupe enfin toute la figure, et une **structure apparaît** : les
clients s'alignent le long d'une diagonale montante. Elle était là avant, on ne
pouvait simplement pas la voir — ni `KMeans` la mesurer.

Il a fallu passer les **deux** axes en échelle logarithmique, et c'est cohérent
avec ce qu'on va faire aux données : `np.log1p` s'applique à **chacune** des
trois variables, pas seulement au montant. Les trois souffrent du même défaut —
quelques individus très loin devant, et tous les autres tassés en bas.

`log1p` et non `log` : il accepte le 0, qu'on a dans `recence` (un client qui a
commandé aujourd'hui).

### Correction 2 : mettre les trois variables au même poids

Le logarithme resserre, mais ne met pas les variables d'accord : après log, la
récence va jusqu'à 5,9 et le montant jusqu'à 11,9. `StandardScaler` ramène
chacune à une moyenne de 0 et un écart-type de 1.

In [ ]:
variables = ["recence", "freq", "montant"]

# 1. log1p ecrase les valeurs extremes (log1p, et non log : il accepte le 0)
# 2. StandardScaler ramene chaque variable a la meme echelle
Xs = StandardScaler().fit_transform(np.log1p(cli[variables]))

print("moyennes apres mise a l'echelle :", Xs.mean(axis=0).round(2))   ## 0
print("ecarts-types                    :", Xs.std(axis=0).round(2))    ## 1

Chaque variable a maintenant une moyenne de 0 et un écart-type de 1 : elles
pèsent le même poids dans le calcul des distances.

> ⚠️ **Sans cette étape, une segmentation se forme sur la variable qui a les
> plus gros nombres.** C'est l'erreur la plus fréquente du clustering, et elle
> ne produit aucun message d'erreur.

## 3. Combien de groupes ?

Vous savez maintenant ce que mesure chacun des deux indicateurs, et à quoi
ressemblent leurs valeurs quand tout va bien. Appliquons-les aux vrais clients
— même code, même balayage de `k` = 2 à 8.

Une seule chose change : le point de départ. Sans aucun groupe (`k` = 1),
l'inertie de ce fichier vaut **1 416**. C'est la dispersion totale à laquelle
on rapportera chaque découpage.

In [ ]:
# k = 1 : aucun groupe, un seul centre. C'est toute la dispersion du fichier
totale = KMeans(n_clusters=1, n_init=10, random_state=42).fit(Xs).inertia_

resultats = []
for k in range(2, 9):   ## de 2 a 8 groupes
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(Xs)
    resultats.append({"k": k,
                      "inertie": round(km.inertia_, 1),
                      "reste_pct": round(100 * km.inertia_ / totale, 1),
                      "silhouette": round(silhouette_score(Xs, km.labels_), 3)})

pd.DataFrame(resultats).set_index("k")

In [ ]:
pd.DataFrame(resultats).set_index("k")["inertie"].plot(marker="o", figsize=(7, 4))
plt.title("La courbe du coude")
plt.ylabel("inertie : dispersion restante")
plt.show()

In [ ]:
pd.DataFrame(resultats).set_index("k")["silhouette"].plot(marker="o", figsize=(7, 4))
plt.title("La silhouette selon k : le decrochage apres 4")
plt.xlabel("nombre de groupes k")
plt.ylabel("silhouette moyenne")
plt.show()

### Lecture du tableau

**L'inertie** tombe de 1 416 à 742 avec deux groupes, puis à 454 avec quatre :
il en reste **32 %**, autrement dit les quatre groupes absorbent plus des deux
tiers de la dispersion. Le gain apporté par chaque groupe supplémentaire, en
points de dispersion absorbée, se lit ainsi :

| passage | gain |
|---|---|
| 2 → 3 | 13,2 pts |
| 3 → **4** | **7,2 pts** |
| 4 → 5 | 4,7 pts |
| 5 → 6 | 3,9 pts |
| 6 → 7 | 2,7 pts |

Le 4ᵉ groupe rapporte encore 7 points ; le 5ᵉ n'en rapporte plus que 4,7, et à
partir de là on descend par petits pas réguliers. **Le coude est à 4** — mais
c'est un coude mou, pas la cassure nette du cas d'école.

**La silhouette** vaut **0,418 pour `k` = 2**, **0,340 pour 3**, **0,321 pour
4**, puis **décroche à 0,301 pour 5** et reste sur ce palier bas : 0,303,
0,304, 0,285. Deux choses à en tirer :

1. C'est `k` = 2 qu'elle **préfère**, et il faut le dire.
2. Mais **`k` = 4 est la dernière valeur au-dessus de 0,32.** Après 4, la
   courbe décroche vers un palier autour de 0,30 dont elle ne remonte jamais.
   Autrement dit, les découpages à 2, 3 et 4 groupes sont sur une même pente ;
   au-delà, on change de régime — on ne fait plus que couper des groupes
   existants en deux, ce qui rapproche les frontières sans rien révéler.

**Les deux indicateurs se rejoignent donc sur une borne haute : 4.** L'inertie
dit « après 4, le gain devient marginal » ; la silhouette dit « après 4, la
séparation se dégrade et ne revient pas ». Aucun des deux ne désigne 4 comme
l'optimum — ils disent qu'**il ne faut pas aller au-delà**, et c'est déjà une
information de cadrage.

Calibrez ces ordres de grandeur avant de conclure : à 0,32 on est dans le
régime **« groupes réels mais qui se chevauchent »** du tableau de la section 1,
et c'est le régime habituel sur des clients réels — le 0,878 du cas d'école
n'existe pas ici. À `k` = 4, **3 % des clients** ont même une silhouette
négative : ils sont du mauvais côté d'une frontière.

### Et pourtant, nous retiendrons 4

Entre 2, 3 et 4 — les trois que les indicateurs laissent ouverts — c'est
l'usage qui tranche. Deux segments, ce sont « les bons » et « les autres » :
aucune action ne s'en déduit. Quatre segments donnent quatre traitements
différents, et c'est ce qu'on demande à une segmentation.

> 💡 **Le nombre de groupes n'est pas une question purement mathématique.**
> Les indicateurs cadrent la décision — ici : pas plus de 4 — l'usage la
> tranche. Ce qu'on doit à son lecteur, c'est de dire que la silhouette
> préférait 2 : pas de le cacher.

## 4. Quand deux variables disent la même chose

Une question demeure : pourquoi la silhouette plafonne-t-elle à 0,32, alors
que le cas d'école atteignait 0,878 ? Une partie de la réponse est dans les
variables elles-mêmes.

In [ ]:
# La correlation se lit sur les variables TELLES QU'ON LES UTILISE, donc en log
np.log1p(cli[variables]).corr().round(2)

`freq` et `montant` corrèlent à **0,77**. Ça n'a rien d'étonnant — un client
qui commande souvent dépense davantage — et c'est précisément le problème :
**ces deux colonnes racontent en grande partie la même histoire.** À côté,
`récence` corrèle à −0,58 et −0,42 : elle, elle apporte autre chose.

### Pourquoi c'est nuisible pour un clustering

`StandardScaler` met les **variables** à poids égal. Il ne met pas les
**informations** à poids égal, et c'est une nuance décisive :

- l'axe *« ce client est un gros client »* est porté par **deux** colonnes, donc
  compté presque **deux fois** dans le calcul des distances ;
- l'axe *« depuis combien de temps ? »* n'est porté que par **une** colonne.

Le nuage s'étire alors dans une direction, comme un ballon de rugby. Or
`KMeans` cherche des paquets **ronds et de tailles comparables** : face à un
nuage allongé sans creux, il fait la seule chose qu'il sache faire — le couper
en rondelles perpendiculaires à son grand axe. Les frontières tombent au milieu
de zones denses, et la silhouette s'effondre : c'est exactement ce que mesure
un score à 0,32.

> **La règle :** avant de segmenter, regardez la matrice de corrélation. Deux
> variables au-dessus de 0,7 pèsent double sans que rien ne vous prévienne.

### Vérifions : les mêmes clients, sans le montant

In [ ]:
duo = ["recence", "freq"]   ## on retire montant, redondant avec freq
Xd = StandardScaler().fit_transform(np.log1p(cli[duo]))

lignes = []
for k in range(2, 9):
    kmd = KMeans(n_clusters=k, n_init=10, random_state=42).fit(Xd)
    lignes.append({"k": k, "silhouette": round(silhouette_score(Xd, kmd.labels_), 3)})

comparaison = pd.DataFrame(lignes).set_index("k")
comparaison["trio RFM"] = pd.DataFrame(resultats).set_index("k")["silhouette"]
comparaison.columns = ["sans montant", "trio RFM"]
comparaison

In [ ]:
comparaison.plot(marker="o", figsize=(7, 4))
plt.title("La silhouette, avec et sans la variable redondante")
plt.xlabel("nombre de groupes k")
plt.ylabel("silhouette moyenne")
plt.show()

**Le changement est net, et il porte précisément sur `k` = 4.**

| k | trio RFM | sans `montant` |
|---|---|---|
| 2 | 0,418 | 0,472 |
| 3 | 0,340 | 0,399 |
| **4** | **0,321** | **0,416** |
| 5 | 0,301 | 0,403 |

Avec le trio, la silhouette **ne fait que descendre** de 2 à 5 : `k` = 4 n'y est
jamais qu'un moindre mal. Sans le montant, elle **remonte** de 0,399 à 0,416 en
passant de 3 à 4 groupes, puis redescend : `k` = 4 devient un vrai **sommet
local**, ce qu'il n'était pas. Le score y gagne 0,095 point — on passe du bas
au haut de la fourchette « groupes qui se chevauchent ».

### L'inertie, elle, ne dit rien ici — et c'est instructif

On serait tenté d'ajouter que la dispersion restante à `k` = 4 tombe de
**32 % à 26 %**. **Ce serait un piège**, et il vaut la peine de le nommer :
après `StandardScaler`, l'inertie totale vaut exactement le nombre de clients
multiplié par le nombre de variables — **472 × 3 = 1 416** pour le trio,
**472 × 2 = 944** sans le montant. Retirer une variable retire une dimension,
donc fait baisser la dispersion mécaniquement, que le découpage soit meilleur
ou non.

C'est le même piège que « l'inertie baisse toujours quand `k` monte »,
transposé aux colonnes. Vérification faite sur ce qui, lui, est comparable —
la netteté du coude, c'est-à-dire le gain du 4ᵉ groupe rapporté à celui du
5ᵉ — le trio donne 7,2 / 4,7 = **1,53** et le duo 7,9 / 5,3 = **1,49** : rien
ne bouge. **Le gain est sur la silhouette seule.**

### Alors pourquoi garder les trois ?

Deux raisons, et il faut les assumer devant un comité :

1. **RFM est un standard.** Vos interlocuteurs connaissent « Recency,
   Frequency, Monetary » ; livrer une segmentation à deux variables demande de
   justifier l'absence de la troisième à chaque réunion.
2. **Le montant porte le récit commercial.** Sans lui, les dormants passent de
   149 à 222 clients, parce qu'on ne distingue plus un gros client endormi d'un
   petit — or c'est exactement sur cette différence que se décide une relance.

> ⚠️ **Et surtout, ne retirez pas `récence`.** Elle donne pourtant la meilleure
> silhouette de toutes (0,472 à `k` = 2) — parce qu'il ne resterait que les deux
> variables corrélées, donc un nuage presque unidimensionnel, trivial à
> découper. **Un bon score obtenu en supprimant de l'information est un mauvais
> score.** C'est le réflexe à garder de cette section.

## 5. Quatre segments, quatre noms

In [ ]:
km = KMeans(n_clusters=4, n_init=10, random_state=42).fit(Xs)
cli["groupe"] = km.labels_   ## un numero de groupe par client

profils = cli.groupby("groupe").agg(
    n=("client_id", "size"),
    recence=("recence", "median"),    ## mediane : robuste, cf. seance 3.1
    freq=("freq", "median"),
    montant=("montant", "median"),
    ca=("montant", "sum"),            ## somme ici : on veut le CA du segment
)
profils["part_ca"] = (100 * profils["ca"] / cli["montant"].sum()).round(1)
profils.round(1)

Chaque ligne se nomme toute seule :

| Profil | Nom |
|---|---|
| 4 jours, 9 commandes, 4 310 € | **les champions** |
| 46 jours, 4 commandes, 1 592 € | **les fidèles** |
| 23 jours, 1 commande, 332 € | **les nouveaux** |
| 186 jours, 1 commande, 429 € | **les dormants** |

Et la colonne qui change tout : **69 champions font 61 % du chiffre
d'affaires**.

C'est le constat de concentration de la séance 2.3 — deux clients irlandais,
22,7 % du CA — retrouvé par un chemin entièrement différent. Trois blocs, trois
méthodes, une même réalité : cette entreprise repose sur une poignée de
comptes.

### Nommer les groupes dans le code, pas seulement dans le commentaire

Le tableau ci-dessus se lit à l'œil. Mais la suite du travail a besoin de
désigner « les dormants » **en Python** — et le faire par leur numéro serait
une erreur, pour une raison qui mérite d'être vue une fois :

> ⚠️ **Les numéros de groupe sont arbitraires.** `KMeans` les attribue dans
> l'ordre où ses centres se sont stabilisés, qui dépend du tirage initial.
> Changez le `random_state`, ajoutez un client au fichier, ou installez une
> autre version de scikit-learn, et le groupe `0` peut devenir le `2`. Écrire
> `groupe == 0` produit alors un code qui tourne encore et qui désigne le
> mauvais segment — encore une erreur silencieuse.

La parade : identifier chaque groupe par **ce qui le caractérise**, jamais par
son numéro. Ces règles-là, elles, sont stables.

In [ ]:
# Chaque nom vient d'un attribut du groupe, jamais de son numero
noms = pd.Series(index=profils.index, dtype="object")
noms[profils["recence"].idxmax()] = "dormants"     ## le plus silencieux
noms[profils["freq"].idxmax()] = "champions"       ## le plus assidu

reste = profils[noms.isna()]                       ## les deux qui restent
noms[reste["montant"].idxmin()] = "nouveaux"       ## a peine arrive
noms[noms.isna()] = "fideles"

profils["nom"] = noms
profils[["nom", "n", "recence", "freq", "montant", "part_ca"]].round(1)

### Le profil de chaque segment, d'un coup d'œil

`km.cluster_centers_` donne la position du centre de chaque groupe, dans
l'espace standardisé. On y lit directement, pour chaque segment, s'il est
au-dessus ou en dessous du client moyen sur chacune des trois variables — et
**0 est exactement le client moyen**.

In [ ]:
centres = pd.DataFrame(km.cluster_centers_, columns=variables)
centres.index = profils["nom"]   ## les noms plutot que les numeros

centres.plot(kind="barh", figsize=(7, 4))
plt.axvline(0, color="black", linewidth=0.8)   ## 0 = le client moyen
plt.title("Le profil de chaque segment, en ecarts-types")
plt.xlabel("ecart au client moyen (0 = la moyenne du fichier)")
plt.ylabel("")
plt.legend(loc="lower right")
plt.show()

Cette figure dit en une image ce que le tableau dit en douze nombres :

- **les champions** sont au-dessus sur `freq` (+1,8) et `montant` (+1,4), et
  très en dessous sur `recence` (−1,4) — ils ont acheté récemment ;
- **les dormants** sont l'exact miroir : `recence` +1,0, tout le reste négatif ;
- **les nouveaux** sont sous la moyenne partout, `montant` compris (−0,8) : ce
  ne sont pas de mauvais clients, ce sont des clients qui viennent d'arriver ;
- **les fidèles** sont proches de 0 sur `recence` : c'est le segment moyen, et
  c'est aussi le plus nombreux.

> 💡 Lisez toujours les centres **après** avoir nommé les groupes. Un centre
> sans nom est une coordonnée ; un centre avec un nom est un client.

In [ ]:
for g in sorted(cli["groupe"].unique()):
    part = cli.query("groupe == @g")
    plt.scatter(part["recence"], part["freq"], alpha=0.6, label=f"groupe {g}")

plt.yscale("log")   ## sans elle, le client a 201 commandes ecrase la figure
plt.xlabel("jours depuis le dernier achat")
plt.ylabel("nombre de commandes (echelle log)")
plt.title("Les quatre segments")
plt.legend()
plt.show()

## 6. Ce qu'on en fait — le plan d'action

On reprend les noms posés à la section 5. Aucun numéro de groupe n'apparaît
dans le code ci-dessous : c'est ce qui le rend rejouable dans six mois, sur un
fichier mis à jour.

In [ ]:
# profils["nom"] a ete construit a partir des attributs, section 5
g_dormants = profils.index[profils["nom"] == "dormants"][0]
g_nouveaux = profils.index[profils["nom"] == "nouveaux"][0]

dormants = cli.query("groupe == @g_dormants")
nouveaux = cli.query("groupe == @g_nouveaux")

print("dormants :", len(dormants), "clients,",
      round(dormants["montant"].sum(), 2), "euros deja depenses")
print("nouveaux :", len(nouveaux), "clients,",
      nouveaux["freq"].median(), "commande en mediane")

**L'arbitrage :** vous avez 5 000 € de budget de relance.

- **Réveiller les 149 dormants.** Ils ont déjà dépensé 74 683 € au total, soit
  500 € chacun en moyenne : ils connaissent le catalogue. Mais leur médiane est
  d'**une seule commande**, et six mois de silence, c'est souvent un client
  déjà parti ailleurs.
- **Convertir les 98 nouveaux.** Une commande en médiane — quelques-uns en ont
  déjà passé jusqu'à quatre. Les faire passer à la deuxième est le geste qui
  transforme un acheteur en client.

Il n'y a pas de réponse mathématique. Il y a une décision à prendre avec des
chiffres — et c'est exactement ce qu'on attend de vous.

> 📌 **Ce que le bloc 4 ne sait pas faire.** Rien ici ne dit qu'une relance
> *marche*. Pour l'établir, il faut relancer un groupe et pas l'autre, puis
> comparer. C'est l'objet du bloc 5.

---

## Ce que vous savez faire maintenant

| Vous voulez... | La commande |
|---|---|
| fabriquer un cas d'école | `make_blobs(n_samples=300, centers=3, random_state=42)` |
| écraser les valeurs extrêmes | `np.log1p(df[variables])` |
| mettre les variables à la même échelle | `StandardScaler().fit_transform(...)` |
| former k groupes | `KMeans(n_clusters=4, n_init=10, random_state=42).fit(X)` |
| l'étiquette de chaque individu | `km.labels_` |
| la compacité des groupes | `km.inertia_` |
| la netteté de la séparation | `silhouette_score(X, km.labels_)` |
| décrire les groupes | `df.groupby("groupe").agg(...)` |
| le profil moyen de chaque groupe | `km.cluster_centers_` |
| repérer deux variables redondantes | `df[variables].corr()` |

## Les deux indicateurs, en une ligne chacun

| | Inertie | Silhouette |
|---|---|---|
| Ce qu'elle mesure | somme des carrés des distances de chaque point au **centre de son groupe** | pour chaque point, l'écart entre sa distance à **son** groupe et sa distance au groupe **voisin** |
| Rôle dans l'algorithme | c'est **exactement ce que `KMeans` minimise** | aucun — c'est un diagnostic ajouté après coup |
| Bon score | **basse** — groupes compacts | **haute**, proche de 1 — groupes bien séparés |
| Étendue | de 0 à l'inertie totale (ici 1 416) | de −1 à 1 |
| Piège | **baisse toujours** quand `k` monte : incomparable d'un `k` à l'autre | aucune, elle est comparable d'un `k` à l'autre |
| Ce qu'on en fait | chercher le **coude** | comparer les `k` directement |

## Supervisé ou non ?

| | Supervisé (4.1 à 4.3) | Non supervisé (4.4) |
|---|---|---|
| On dispose de... | une cible connue | rien d'autre que les variables |
| On mesure la qualité par... | l'erreur sur un jeu de test | la cohérence des groupes, et l'usage qu'on en fait |
| La bonne réponse... | existe | **n'existe pas** — il y a des découpages plus ou moins utiles |

## Les trois phrases à retenir

1. **Standardiser n'est pas optionnel.** Sans mise à l'échelle, les groupes se
   forment sur la variable qui a les plus gros nombres — ici le montant, et
   deux clients se retrouvent seuls dans leur segment.

2. **Le nombre de groupes n'est pas une question purement mathématique.** La
   silhouette préfère 2 (0,418 contre 0,321) ; l'usage commercial en demande 4.
   Les deux se défendent, et c'est à vous d'arbitrer.

3. **Une segmentation sans nom ni budget ne sert à rien.** « Groupe 0 » n'est
   pas un livrable ; « 69 champions qui font 61 % du chiffre d'affaires » en
   est un.